In [19]:
!pip install -q sentence-transformers faiss-cpu pymupdf google-genai beautifulsoup4 requests


In [20]:
import os
from google import genai
from getpass import getpass

api_key = getpass("Enter your Gemini API Key: ")

os.environ["GOOGLE_API_KEY"] = api_key

client = genai.Client(api_key=api_key)

Enter your Gemini API Key: ··········


In [21]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

print("Uploaded:", pdf_path)

Saving Anuresh Pattanayak cv.docx to Anuresh Pattanayak cv (1).docx
Uploaded: Anuresh Pattanayak cv (1).docx


In [22]:
import fitz

def load_resume(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

resume_text = load_resume(pdf_path)
print("\nResume Loaded Successfully!")


Resume Loaded Successfully!


In [23]:
import re

def preprocess_resume(text):
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\+?\d[\d\s\-]{8,}', '', text)
    return text

resume_text = preprocess_resume(resume_text)

In [24]:
def create_chunks(text):
    sections = re.split(r'\n+', text)
    chunks = []

    for sec in sections:
        sec = sec.strip()
        if len(sec) > 40:
            chunks.append(sec)

    return chunks

chunks = create_chunks(resume_text)
print("Total Chunks:", len(chunks))

Total Chunks: 24


In [25]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def embed_chunks(chunks):
    return embed_model.encode(chunks)

embeddings = embed_chunks(chunks)
print("Embeddings Created")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings Created


In [26]:
import faiss
import numpy as np

def build_vector_store(embeddings):
    dimension = len(embeddings[0])
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings))
    return index

index = build_vector_store(embeddings)
print("Vector DB Ready")

Vector DB Ready


In [27]:
def query_vector_store(index, chunks, query, k=5):
    query_embedding = embed_model.encode([query])
    distances, indices = index.search(np.array(query_embedding), k)
    return [chunks[i] for i in indices[0]]

In [28]:
import requests
from bs4 import BeautifulSoup

def scrape_job_description(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    return soup.get_text(separator=" ", strip=True)

In [29]:
def ask_gemini(prompt):
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )
        return response.text
    except Exception as e:
        return f"Error: {e}"

In [30]:
def extract_skills(text):
    prompt = f"""
    Extract ONLY technical skills from the text below.
    Return strictly as a Python list.

    Text:
    {text[:3000]}
    """
    return ask_gemini(prompt)

In [31]:
def compare_skills(resume_skills, job_skills):
    prompt = f"""
    Compare these:

    Resume Skills:
    {resume_skills}

    Job Skills:
    {job_skills}

    Return ONLY:

    Matching Skills: []
    Missing Skills: []
    Match Percentage: %
    """
    return ask_gemini(prompt)

In [32]:
def final_decision(result):
    prompt = f"""
    Based on:

    {result}

    If match percentage >= 70% → SELECTED
    Else → NOT SELECTED

    Output ONLY:
    Decision: SELECTED or NOT SELECTED
    """
    return ask_gemini(prompt)

In [33]:
job_url = input("\nEnter Job URL: ")

print("\nScraping Job Description...")
job_text = scrape_job_description(job_url)


Enter Job URL: https://www.naukri.com/job-listings-software-engineer-paramarsh-informatics-delhi-ncr-0-to-5-years-200226014559?src=jobsearchDesk&sid=17743367950208436&xp=3&px=1&nignbevent_src=jobsearchDeskGNB

Scraping Job Description...


In [34]:
relevant_chunks = query_vector_store(index, chunks, "skills experience technologies", k=5)
filtered_resume_text = " ".join(relevant_chunks)

In [35]:
print("\nExtracting Resume Skills...")
resume_skills = extract_skills(filtered_resume_text)
print("Resume Skills:", resume_skills)


Extracting Resume Skills...
Resume Skills: ```python
    ["UI/UX", "responsive navigation menu", "interactive product cards", "cross-device compatibility", "mobile responsiveness"]
    ```


In [36]:
print("\nExtracting Job Skills...")
job_skills = extract_skills(job_text)
print("Job Skills:", job_skills)



Extracting Job Skills...
Job Skills: I need the text to extract the skills from. Please provide the text.

If the text is empty, the result would be:

```python
[]
```


In [37]:
print("\nPlease paste the Job Description here (Press Enter twice to stop):")
job_lines = []
while True:
    line = input()
    if line == "":
        break
    job_lines.append(line)

job_text = "\n".join(job_lines)
print("\nJob Description captured.")


Please paste the Job Description here (Press Enter twice to stop):
Develop and maintain web applications using Java, HTML, CSS, and JavaScript 	•	Work with backend technologies like Node.js / Express and MySQL 	•	Design and implement database structures and queries 	•	Collaborate with team members to build scalable and efficient systems 	•	Debug, test, and optimize application performance 	•	Integrate APIs and third-party services (e.g., authentication, book APIs) 	•	Participate in project development such as Airline Reservation System and Bookique


Job Description captured.


In [38]:
print("\nExtracting Job Skills...")
job_skills = extract_skills(job_text)
print("Job Skills:", job_skills)


Extracting Job Skills...
Job Skills: ```python
[
    "Java",
    "HTML",
    "CSS",
    "JavaScript",
    "Node.js",
    "Express",
    "MySQL",
    "Database design",
    "Database queries",
    "Debugging",
    "Testing",
    "Performance optimization",
    "API integration"
]
```


In [39]:
print("\nComparing Skills...")
comparison = compare_skills(resume_skills, job_skills)
print("\n", comparison)

print("\nFinal Decision...")
decision = final_decision(comparison)
print(decision)


Comparing Skills...

 Matching Skills: []
Missing Skills: ["Java", "HTML", "CSS", "JavaScript", "Node.js", "Express", "MySQL", "Database design", "Database queries", "Debugging", "Testing", "Performance optimization", "API integration"]
Match Percentage: 0%

Final Decision...
Decision: NOT SELECTED
